# 07 · The team harness

Everything so far was about one agent: what it can touch, what it knows, what
it may not do.

This notebook is about the layer above. DeepAgents gives you an excellent
harness for **an** agent. It has nothing to say about six of them working on
the same thing at once.

That gap is what `apps/control_plane/` fills, and the shape of the answer is
worth studying because it is the same shape as a middleware stack — just one
level up.

In [1]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

repo root: /Users/aseem/Documents/hubbleflow/standalone-projects/agentic-crew


## What is missing when you have six

| Question | DeepAgents | Here |
|---|---|---|
| What are we working on? | — | `domain/project.py` |
| How many of each role? | — | `domain/caps.py` |
| Who decides? | — | the EM's tool catalogue |
| Where does work live? | one agent's filesystem | one volume, `subPath` per project |
| How do they hear each other? | — | one Redis channel per project |
| What happened? | — | the recorder + Loki |

None of that is a criticism. A harness for one agent should not have opinions
about org charts.

## Four layers, one job each

The whole control plane is arranged so that each file has exactly one kind of
thing in it.

In [2]:
for layer, blurb in [
    ("domain",   "rules. no I/O at all"),
    ("ports",    "contracts. what is needed, never how"),
    ("adapters", "I/O. no rules"),
    ("api",      "HTTP. no decisions"),
]:
    files = sorted(p.name for p in (ROOT / "apps/control_plane" / layer).glob("*.py")
                   if p.name != "__init__.py")
    print(f"  {layer:<10} {blurb:<32} {', '.join(files)}")
print(f"  {'service.py':<10} {'use cases · the only layer that knows both'}")

  domain     rules. no I/O at all             caps.py, project.py
  ports      contracts. what is needed, never how events.py, runtime.py, sandbox.py, store.py
  adapters   I/O. no rules                    fake_runtime.py, k8s_runtime.py, k8s_sandbox.py, memory_store.py, redis_events.py
  api        HTTP. no decisions               app.py, deps.py, routes.py, schemas.py
  service.py use cases · the only layer that knows both


The test for whether this is real: can the rules be exercised with no
infrastructure?

In [3]:
import subprocess, sys, time

start = time.perf_counter()
out = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "--no-header"],
                     cwd=ROOT, capture_output=True, text=True)
elapsed = time.perf_counter() - start

print(out.stdout.strip().splitlines()[-1])
print(f"in {elapsed:.1f}s — no cluster, no Redis, no Docker")

55 passed, 1 warning in 0.41s
in 0.9s — no cluster, no Redis, no Docker


That is the entire argument for the split. Those tests cover the spawn caps,
the naming rules, the refusal messages and the whole HTTP surface, and they run
faster than a container can start.

## Ports: the seam

A port is a contract with no implementation. Here is the one that decides where
agents run.

In [4]:
src = (ROOT / "apps/control_plane/ports/runtime.py").read_text()
start = src.index("class AgentRuntime")
print(src[start:].rstrip())

class AgentRuntime(Protocol):
    """Somewhere agents can be run.

    Implementations live in ``adapters/``:

    * ``k8s_runtime`` — creates a Job and lets the cluster own the lifecycle:
      placement, retry, completion, and TTL cleanup. This is the real one.
    * ``fake_runtime`` — an in-memory stand-in so the spawn rules can be tested
      without a cluster.

    There is deliberately no Docker implementation. Running containers by hand
    means re-implementing retry, cleanup and placement in application code,
    which is the thing this project moved away from.
    """

    async def launch(self, spec: AgentSpec) -> AgentHandle:
        """Ask for an agent. Returns as soon as the request is accepted.

        This does not wait for the agent to be ready. Under Kubernetes it is
        one API call and then the cluster's problem.
        """
        ...

    async def status(self, handle: AgentHandle) -> AgentStatus:
        """Where the agent has got to, and why if it is not 

Two implementations satisfy it. One creates Kubernetes Jobs. One is a
dictionary.

In [5]:
from apps.control_plane.adapters.fake_runtime import FakeAgentRuntime
from apps.control_plane.adapters.k8s_runtime import KubernetesAgentRuntime
from apps.control_plane.ports.runtime import AgentRuntime

for impl in (KubernetesAgentRuntime, FakeAgentRuntime):
    print(f"  {impl.__name__:<26} satisfies AgentRuntime: {isinstance(impl(), AgentRuntime)}")

  KubernetesAgentRuntime     satisfies AgentRuntime: True
  FakeAgentRuntime           satisfies AgentRuntime: True


Nothing above the port can tell them apart, which is why the tests above are
possible at all.

## A rule, with no infrastructure in it

`domain/project.py` is the whole of "a chat is called New Project until the EM
knows what it is".

In [6]:
from apps.control_plane.domain.project import PROVISIONAL_NAME, NamingError, Project

p = Project(id="proj-demo")
print(f"  new project        name={p.name!r}  is_named={p.is_named}")

named = p.rename("  Rate   limiting for the public API ")
print(f"  after rename       name={named.name!r}  is_named={named.is_named}")

for bad in ("", "New Project", "x" * 80):
    try:
        p.rename(bad)
    except NamingError as e:
        print(f"  rejected {bad[:14]!r:<18} {e}")

print(f"\n  original unchanged: {p.name!r}   (frozen · transitions return new instances)")

  new project        name='New Project'  is_named=False
  after rename       name='Rate limiting for the public API'  is_named=True
  rejected ''                 a project name cannot be empty
  rejected 'New Project'      'New Project' is the placeholder, not a name. Name the project after what is being built.
  rejected 'xxxxxxxxxxxxxx'   project name is 80 characters; the limit is 60

  original unchanged: 'New Project'   (frozen · transitions return new instances)


No database, no cluster, no mock. A rule you can read in one file and check in
one line.

## Who decides who answers

The founder talks to two roles. Which one handles a given message is the EM's
call, and that decision is not in code — it is in a skill.

In [7]:
skill = (ROOT / "skills/roles/engineering-manager/scoping-a-request/SKILL.md").read_text()
start = skill.index("## Who answers?")
print(skill[start:skill.index("## Spawning")].rstrip())

## Who answers?

You and the Product Manager are the two roles the founder talks to. Decide
deliberately:

**Answer yourself** when the request is technically clear and the work is
obvious. Bringing in a PM to clarify an unambiguous request wastes a turn and
makes the crew look slow.

**Spawn the PM** when the request is ambiguous about *what the user needs*,
rather than about how to build it. "Make onboarding better" needs a PM.
"Add a /healthz endpoint" does not.


This is a judgement call with no clean rule behind it, so it belongs in
instructions the model reads, not in a branch. The things with clean rules —
how many agents, what a project may be called — are in `domain/`.

That division is the useful one: **rules in code, judgement in skills.**

## Recording what happened

One subtlety that cost a debugging session and is worth repeating.

Agents publish events from inside their own pods. They cannot reach the control
plane's store. So if every publisher also recorded, a project's history would
contain only what the control plane itself said — every tool call and every
agent message would be lost.

In [8]:
src = (ROOT / "apps/control_plane/service.py").read_text()
start = src.index("    async def _emit")
print(src[start:].rstrip())

    async def _emit(self, event: Event) -> None:
        """Publish onto the bus. Recording is somebody else's job.

        Deliberately not written to the store here. Agents publish from inside
        their own pods and cannot reach the store at all, so if each publisher
        also recorded, the history would hold control-plane events and nothing
        else. :meth:`record` is the single writer, and it sees both.
        """
        await self._events.publish(event)

    async def record(self) -> None:
        """Write every event on the bus into the store. Runs forever.

        Started once at boot. This is what makes a project a founder reopens
        tomorrow read the same as the one they watched today.
        """
        async for event in self._events.subscribe_all():
            try:
                await self._store.append_event(event)
            except Exception:
                # A history gap is bad; a recorder that dies and leaves every
                # later even

One subscriber, on a pattern that matches every project, writing everything
down. Single writer, and it sees both sides.

## What you now know

1. DeepAgents harnesses one agent; a *team* needs projects, caps, a shared
   workspace, a bus, and a record. That is `apps/control_plane/`.
2. Four layers, one concern each. The proof it is real: the rules test in under
   a second with nothing running.
3. A port has two implementations — Kubernetes and a dictionary — and nothing
   above it can tell which is in use.
4. Rules in code, judgement in skills.
5. One recorder writes history, because the publishers cannot.

Next: all of it at once, on a real cluster.